# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`

This notebook provides a guided exploration of the FAIR^2 dataset on non-classical 21-hydroxylase deficiency-associated infertility using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields (do not treat as dict, use properties)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Dataset published: {dataset.metadata.datePublished}")
print(f"Dataset identifier: {dataset.metadata.identifier}")
print(f"Dataset license: {dataset.metadata.license}")
print("Keywords:")
pprint.pprint(dataset.metadata.keywords)

## 2. Data Overview

Review available record sets, their fields, and corresponding `@id` values for referencing entities throughout the dataset.

In [ ]:
# Discover available record sets
record_sets = list(dataset.record_sets())
print("Available record sets (with their @id):")
for rs in record_sets:
    print(f"- Record Set name: {rs.name}, @id: {rs.id}")
    print("  Fields in this record set:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, type: {field.dataType}")
    print()

## 3. Data Extraction

Load data from selected record sets into DataFrames for further analysis, referencing entities using their `@id`s.

In [ ]:
# List record set IDs
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecord Set '@id': {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"First entries:")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below we show: (1) field selection by `@id`, (2) filtering and normalization, (3) grouping.

In [ ]:
# Select a record set and fields for EDA
# For demonstration, take the first record set
if len(record_sets) == 0:
    print('No record sets found in this dataset.')
else:
    demo_rs = record_sets[0]
    demo_rs_id = demo_rs.id
    df_demo = dataframes[demo_rs_id]
    print(f"Using Record Set '@id': {demo_rs_id} ({demo_rs.name})")

    # List numeric fields by checking their dataType
    numeric_field_ids = [field.id for field in demo_rs.fields if field.dataType in ['schema:Float', 'schema:Integer', 'schema:Number']]

    if len(numeric_field_ids) == 0:
        print('No numeric fields detected in this record set for EDA.')
    else:
        numeric_field_id = numeric_field_ids[0]
        print(f"Analyzing numeric field with @id: {numeric_field_id}")

        # Filter for values greater than a threshold
        threshold = 10
        filtered_df = df_demo[df_demo[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric column
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a non-numeric field, e.g., the first non-numeric
        non_numeric_field_ids = [field.id for field in demo_rs.fields if field.dataType not in ['schema:Float', 'schema:Integer', 'schema:Number']]
        if len(non_numeric_field_ids) > 0:
            group_field_id = non_numeric_field_ids[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"Grouped data by '{group_field_id}' (mean of '{numeric_field_id}'):")
                print(grouped_df.head())

## 5. Visualization

Visualize selected numeric field distributions and relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plot histogram for the previously selected numeric field
if len(record_sets) > 0 and len(numeric_field_ids) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_demo[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Record Set {demo_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, show relationship
    if len(non_numeric_field_ids) > 0 and group_field_id in df_demo.columns:
        plt.figure(figsize=(7, 5))
        sns.boxplot(x=df_demo[group_field_id], y=df_demo[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, inspect, and analyze a multi-table clinical genotype–phenotype dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`s. You can extend this exploration further by inspecting other record sets and their fields, adapting EDA and visualization for more granular or domain-specific analyses.

For further information or to cite this dataset, use the published identifier: `10.71728/senscience.cbsv-djdb`.